In [1]:
# ----------------------------------------------------- #
# Experiments set to run

experiments_to_run = [
    #"asqa_gemma_cb",
    #"asqa_gemma_splade",
    "asqa_gemma_bm25",

    #"asqa_llama_cb",
    #"asqa_llama_splade",
    #"asqa_llama_bm25",

    #"asqa_mistral_cb",
    #"asqa_mistral_splade",
    #"asqa_mistral_bm25"
]

# Run LLMEVal evaluation

run_eval = False

# ----------------------------------------------------- #
# Checks to make sure retrievers load

retriever_checks = [
    #"splade",
    #"bm25"
]

# Checks to make sure models load
model_checks = [
    #"gemma",
    #"llama",
    #"mistral"
]

# "local": Create HuggingFace cache locally in Colab
# "drive": Create cache in Google Colab (Uses up space on drive!)
# "{PATH}": provide a path in which to create the cache (deleted upon runtime end)
loc_hf_cache = "local"

# ----------------------------------------------------- #
# Debug stuff:

# Generate frozen requirements document
freeze_requirements = False

In [2]:
# Mount Google Drive.
# BERGEN files and our HuggingFace dataset caches are stored there

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Set OS environment paths

if loc_hf_cache == "drive":
    hf_cache = "/content/drive/MyDrive/hf_cache"
elif loc_hf_cache == "local":
    hf_cache = "/var/tmp/hf_cache"
else:
    hf_cache = loc_hf_cache

import os
from google.colab import userdata

os.environ["HF_DATASETS_CACHE"] = hf_cache
os.environ["HF_HOME"] = hf_cache
os.environ["TRANSFORMERS_CACHE"] = hf_cache

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["JVM_PATH"] = "/usr/lib/jvm/java-21-openjdk-amd64/lib/server/libjvm.so"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

# Load HuggingFace token

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# Change execution directory to where bergen is located in our Google Drive

os.chdir('/content/drive/MyDrive/Colab Notebooks/bergen')

In [4]:
# Install base requirements.
# Using this updated file allows pip to work out all of the dependency issues.

!pip install -r requirements_josiah.txt

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.1/166.1 MB 14.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 96.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 8.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 64.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of pyserini to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.8/164.8 MB 12.2 MB/s eta 0:00:00
  Installing build 

In [5]:
# Generate frozen requirements file (file that shows all of the packages
# installed and their versions).

if freeze_requirements:
    !pip freeze > frozen_requirements_2.txt


Hugging Face login

In [6]:
# Login to HuggingFace via your 'HF_TOKEN' user token.
# This is necessary as some of the model repos are gated and require
# permission to use.
#
# It is necessary to store your access token in your Colab Secrets
# to run this notebook.

from huggingface_hub import login
login()

In [7]:
# Install JVM 21 (resolves dependencies with some of the models)

!apt-get update -qq
!apt-get install -y openjdk-21-jdk-headless

# After installing JVM 21, double-check that both v. 17 and 21 are there.

print("\nVersions of JVM installed:")
!find /usr/lib/jvm -name "libjvm.so"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/jammy/InRelease  Could not wait for server fd - select (11: Resource temporarily unavailable) [IP: 185.125.190.80 443]
W: Failed to fetch https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu/dists/jammy/InRelease  Could not connect to ppa.launchpadcontent.net:443 (185.125.190.80), connection timed out [IP: 185.125.190.80 443]
W: Failed to fetch https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu/dists/jammy/InRelease  Unable to connect to ppa.launchpadcontent.net:443: [IP: 185.125.190.80 443]
W: Some index files failed to download. They have been ignored, or old ones used instead.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages wi

In [36]:
!pip uninstall vllm

Found existing installation: vllm 0.15.0
Uninstalling vllm-0.15.0:
  Would remove:
    /usr/local/bin/vllm
    /usr/local/lib/python3.12/dist-packages/vllm-0.15.0.dist-info/*
    /usr/local/lib/python3.12/dist-packages/vllm/*
Proceed (Y/n)? y
  Successfully uninstalled vllm-0.15.0


In [9]:
# Check if BM25 correctly loaded.

if "bm25" in retriever_checks:
    !JAVA_HOME="/usr/lib/jvm/java-21-openjdk-amd64" \
    JVM_PATH="/usr/lib/jvm/java-21-openjdk-amd64/lib/server/libjvm.so" \
    PYTHONPATH="/content/drive/MyDrive/Colab Notebooks/bergen" \
    python3 -c "from models.retrievers.bm25 import Splade; print('BM25 works')"

In [10]:
# Check if Splade correctly loaded.

if "splade" in retriever_checks:
    !JAVA_HOME="/usr/lib/jvm/java-21-openjdk-amd64" \
    JVM_PATH="/usr/lib/jvm/java-21-openjdk-amd64/lib/server/libjvm.so" \
    PYTHONPATH="/content/drive/MyDrive/Colab Notebooks/bergen" \
    python3 -c "from models.retrievers.splade import Splade; print('Splade works')"

In [11]:
# Check if Gemma 3 correctly loaded.
if "gemma" in model_checks:
    from transformers import AutoTokenizer

    AutoTokenizer.from_pretrained("google/gemma-3-1b-it")
    print("Gemma works")

# Check if Llama 3.2 correctly loaded.
if "llama" in model_checks:
    from transformers import AutoTokenizer

    AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
    print("Llama works")

# Check if Mistral correctly loaded.
if "mistral" in model_checks:
    from transformers import AutoTokenizer

    AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")
    print("Mistral works")

# Experiments using Gemma 1B

## Experiment 1) Open book w/ Splade
This experiment tests **how well does a small LLM answer questions when given retrieved documents**. We are using:

*   kill_nq from BERGEN’s built-in dataset configurations, a question answering dataset with linked documents from Wikipedia.
*   bm25 retriever, testing if simple non-neural keyword matching finds good enough documents.
*   debertav3 reranker, to see if reordering the retrieved results is of any use



In [12]:
import time

In [13]:
# Dataset: ASQA | Model: Gemma 3 | Retriever: None

if "asqa_gemma_cb" in experiments_to_run:
    ti = time.time()
    !PYTHONPATH="/content/drive/MyDrive/Colab Notebooks/bergen" python3 bergen.py dataset=asqa_Karpukhin generator=gemma-3-1b prompt=asqa +run_name=asqa_gemma_cb

    tf = time.time()
    t_total = tf - ti
    print("Total running time:", t_total)

In [14]:
# Dataset: ASQA | Model: Gemma 3 | Retriever: Splade-v3

if "asqa_gemma_splade" in experiments_to_run:
    ti = time.time()
    !PYTHONPATH="/content/drive/MyDrive/Colab Notebooks/bergen" python3 bergen.py dataset=asqa_Karpukhin retriever=spladev3-fatboy reranker=debertav3 generator=gemma-3-1b prompt=asqa +run_name=asqa_gemma_splade

    tf = time.time()
    t_total = tf - ti
    print("Total running time:", t_total)

In [15]:
# Dataset: ASQA | Model: Gemma 3 | Retriever: BM25

if "asqa_gemma_bm25" in experiments_to_run:
    ti = time.time()
    !PYTHONPATH="/content/drive/MyDrive/Colab Notebooks/bergen" python3 bergen.py dataset=asqa_Karpukhin retriever=bm25 reranker=debertav3 generator=gemma-3-1b prompt=asqa +run_name=asqa_gemma_bm25

    tf = time.time()
    t_total = tf - ti
    print("Total running time:", t_total)

In [16]:
# Dataset: ASQA | Model: Llama 3.2 | Retriever: None

if "asqa_llama_cb" in experiments_to_run:
    ti = time.time()
    !PYTHONPATH="/content/drive/MyDrive/Colab Notebooks/bergen" python3 bergen.py dataset=asqa_Karpukhin generator=llama-3.2-3b-instruct prompt=asqa +run_name=asqa_llama_cb

    tf = time.time()
    t_total = tf - ti
    print("Total running time:", t_total)

In [17]:
# Dataset: ASQA | Model: Llama 3.2 | Retriever: Splade-v3

if "asqa_llama_splade" in experiments_to_run:
    ti = time.time()
    !PYTHONPATH="/content/drive/MyDrive/Colab Notebooks/bergen" python3 bergen.py dataset=asqa_Karpukhin retriever=spladev3-fatboy reranker=debertav3 generator=llama-3.2-3b-instruct prompt=asqa +run_name=asqa_llama_splade

    tf = time.time()
    t_total = tf - ti
    print("Total running time:", t_total)

In [18]:
# Dataset: ASQA | Model: Llama 3.2 | Retriever: BM25

if "asqa_llama_bm25" in experiments_to_run:
    ti = time.time()
    !PYTHONPATH="/content/drive/MyDrive/Colab Notebooks/bergen" python3 bergen.py dataset=asqa_Karpukhin retriever=bm25 reranker=debertav3 generator=llama-3.2-3b-instruct prompt=asqa +run_name=asqa_llama_bm25

    tf = time.time()
    t_total = tf - ti
    print("Total running time:", t_total)

In [19]:
# Dataset: ASQA | Model: Mistral | Retriever: None

if "asqa_mistral_cb" in experiments_to_run:
    ti = time.time()
    !PYTHONPATH="/content/drive/MyDrive/Colab Notebooks/bergen" python3 bergen.py dataset=asqa_Karpukhin prompt=asqa generator=mistral-7b-chat +run_name=asqa_mistral_cb

    tf = time.time()
    t_total = tf - ti
    print("Total running time:", t_total)

In [20]:
# Dataset: ASQA | Model: Mistral | Retriever: Splade-v3

if "asqa_mistral_splade" in experiments_to_run:
    ti = time.time()
    !PYTHONPATH="/content/drive/MyDrive/Colab Notebooks/bergen" python3 bergen.py dataset=asqa_Karpukhin retriever=spladev3-fatboy reranker=debertav3 generator=mistral-7b-chat prompt=asqa +run_name=asqa_mistral_splade

    tf = time.time()
    t_total = tf - ti
    print("Total running time:", t_total)

In [21]:
# Dataset: ASQA | Model: Mistral | Retriever: BM25

if "asqa_mistral_bm25" in experiments_to_run:
    ti = time.time()
    !PYTHONPATH="/content/drive/MyDrive/Colab Notebooks/bergen" python3 bergen.py dataset=asqa_Karpukhin retriever=bm25 reranker=debertav3 generator=mistral-7b-chat prompt=asqa +run_name=asqa_mistral_bm25

    tf = time.time()
    t_total = tf - ti
    print("Total running time:", t_total)

In [40]:
if run_eval:
    !pip install vllm==0.11.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.2/438.2 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.0/180.0 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 113.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 50.7 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 80.10.2
    Uninstalling setuptools-80.10.2:
     

In [41]:
if run_eval:
    ti = time.time()
    !python3 evaluate.py --experiments_folder experiments/ --llm_batch_size 32 --split 'dev' --llm vllm_SOLAR-107B

    tf = time.time()
    t_total = tf - ti
    print("Total running time:", t_total)

Streaming output truncated to the last 5000 lines.
<string>:1: SyntaxWarning: invalid escape sequence '\ '
<string>:1: SyntaxWarning: invalid escape sequence '\ '
<string>:1: SyntaxWarning: invalid escape sequence '\ '
<string>:1: SyntaxWarning: invalid escape sequence '\ '
<string>:1: SyntaxWarning: invalid escape sequence '\ '
<string>:1: SyntaxWarning: invalid escape sequence '\ '
<string>:1: SyntaxWarning: invalid escape sequence '\ '
<string>:1: SyntaxWarning: invalid escape sequence '\ '
<string>:1: SyntaxWarning: invalid escape sequence '\ '
<string>:1: SyntaxWarning: invalid escape sequence '\ '
<string>:1: SyntaxWarning: invalid escape sequence '\ '
<string>:1: SyntaxWarning: invalid escape sequence '\ '
<string>:1: SyntaxWarning: invalid escape sequence '\ '
<string>:1: SyntaxWarning: invalid escape sequence '\ '
<string>:1: SyntaxWarning: invalid escape sequence '\ '
<string>:1: SyntaxWarning: invalid escape sequence '\ '
<string>:1: SyntaxWarning: invalid escape sequence '\

In [25]:
# After the test run, make sure that we got some outputs
# by opening the eval_dev_out.json and peeking at the LLM's
# results.
'''
import json

path = "/content/drive/MyDrive/bergen/experiments/first_test/eval_dev_out.json"

with open(path) as f:
    data = json.load(f)

for ex in data[:5]:
    print("Q:", ex["question"])
    print("Prediction:", ex["response"])
    print("Gold:", ex["label"])
    print("EM:", ex["EM"], "F1:", ex["F1"])
    print("-" * 80)'''

'\nimport json\n\npath = "/content/drive/MyDrive/bergen/experiments/first_test/eval_dev_out.json"\n\nwith open(path) as f:\n    data = json.load(f)\n\nfor ex in data[:5]:\n    print("Q:", ex["question"])\n    print("Prediction:", ex["response"])\n    print("Gold:", ex["label"])\n    print("EM:", ex["EM"], "F1:", ex["F1"])\n    print("-" * 80)'

Enhancing my RAG pipeline results by changing the prompt to model:

In [26]:
#!cat "/content/drive/MyDrive/Colab Notebooks/bergen/config/prompt/nq.yaml"

New test using prompt:

```
system: "You are a QA system. Answer with ONLY the exact answer. Do not explain. Do not add extra words. Output only a single word or short phrase."

user: "Background:\n{docs}\n\nQuestion: {question}\nAnswer:"

system_without_docs: "Answer with ONLY the exact answer. Do not explain. Output only a single word or short phrase."

user_without_docs: "Question: {question}\nAnswer:"
```



In [27]:
#!JAVA_HOME="/usr/lib/jvm/java-21-openjdk-amd64" JVM_PATH="/usr/lib/jvm/java-21-openjdk-amd64/lib/server/libjvm.so" PYTHONPATH="/content/drive/MyDrive/Colab Notebooks/bergen" python3 bergen.py dataset=kilt_nq prompt=nq retriever=bm25 reranker=debertav3 generator=gemma-3-1b +run_name=strict_prompt_test

Test results:

In [28]:
#!cat "/content/drive/MyDrive/Colab Notebooks/bergen/experiments/first_test/eval_dev_metrics.json"

In [29]:
#!cat "/content/drive/MyDrive/Colab Notebooks/bergen/experiments/strict_prompt_test/eval_dev_metrics.json"

Compare examples:

In [30]:
'''import json

base_path = "/content/drive/MyDrive/Colab Notebooks/bergen/experiments/first_test/eval_dev_out.json"
strict_path = "/content/drive/MyDrive/Colab Notebooks/bergen/experiments/strict_prompt_test/eval_dev_out.json"

with open(base_path) as f:
    base = json.load(f)

with open(strict_path) as f:
    strict = json.load(f)

for i in range(5):
    print("Q:", base[i]["question"])
    print("Old:", base[i]["response"])
    print("New:", strict[i]["response"])
    print("Gold:", base[i]["label"])
    print("-" * 80)'''

'import json\n\nbase_path = "/content/drive/MyDrive/Colab Notebooks/bergen/experiments/first_test/eval_dev_out.json"\nstrict_path = "/content/drive/MyDrive/Colab Notebooks/bergen/experiments/strict_prompt_test/eval_dev_out.json"\n\nwith open(base_path) as f:\n    base = json.load(f)\n\nwith open(strict_path) as f:\n    strict = json.load(f)\n\nfor i in range(5):\n    print("Q:", base[i]["question"])\n    print("Old:", base[i]["response"])\n    print("New:", strict[i]["response"])\n    print("Gold:", base[i]["label"])\n    print("-" * 80)'

## Experiment 2) Open book w/ bm25 & ASQA
This experiment tests **how well does a small LLM answer questions when given retrieved documents**. We are using:

*   ASQA from BERGEN’s built-in dataset configurations, a long-form question answering dataset focusing on ambiguous questions.
*   bm25 retriever, testing if simple non-neural keyword matching finds good enough documents.
*   debertav3 reranker, to see if reordering the retrieved results is of any use

In [31]:
#!JAVA_HOME="/usr/lib/jvm/java-21-openjdk-amd64" \
#JVM_PATH="/usr/lib/jvm/java-21-openjdk-amd64/lib/server/libjvm.so" \
#PYTHONPATH="/content/drive/MyDrive/Colab Notebooks/bergen" \
#python3 -c "from models.retrievers.bm25 import BM25; print('BM25 works')"

In [32]:
'''!JAVA_HOME="/usr/lib/jvm/java-21-openjdk-amd64" \
JVM_PATH="/usr/lib/jvm/java-21-openjdk-amd64/lib/server/libjvm.so" \
PYTHONPATH="/content/drive/MyDrive/Colab Notebooks/bergen" \
python3 bergen.py dataset=asqa prompt=basic retriever=bm25 reranker=debertav3 generator=gemma-3-1b +run_name=asqa_basic_test'''

'!JAVA_HOME="/usr/lib/jvm/java-21-openjdk-amd64" JVM_PATH="/usr/lib/jvm/java-21-openjdk-amd64/lib/server/libjvm.so" PYTHONPATH="/content/drive/MyDrive/Colab Notebooks/bergen" python3 bergen.py dataset=asqa prompt=basic retriever=bm25 reranker=debertav3 generator=gemma-3-1b +run_name=asqa_basic_test'

In [33]:
# Dataset: ASQA | Generator: Mistral 7B | Retriever: Splade
'''
if "asqa_mistral_splade" in experiments_to_run:
    !PYTHONPATH="/content/drive/MyDrive/Colab Notebooks/bergen" python3 bergen.py dataset=asqa retriever=spladev3 reranker=debertav3 generator=vllm_mistral-7b-chat +run_name=asqa_mistral_splade'''

'\nif "asqa_mistral_splade" in experiments_to_run:\n    !PYTHONPATH="/content/drive/MyDrive/Colab Notebooks/bergen" python3 bergen.py dataset=asqa retriever=spladev3 reranker=debertav3 generator=vllm_mistral-7b-chat +run_name=asqa_mistral_splade'

## Experiment 3) Open book w/ bm25 & HotPotQA
This experiment tests **how well does a small LLM answer questions when given retrieved documents**. We are using:

*   HotPotQA from BERGEN’s built-in dataset configurations, a multi-hop question answering dataset where answering the question requires combining information from multiple docu- ments.
*   bm25 retriever, testing if simple non-neural keyword matching finds good enough documents.
*   debertav3 reranker, to see if reordering the retrieved results is of any use

## Experiment 4) Closed book w/ bm25 & NQ
This experiment tests **how well does a small LLM answer questions when not given retrieved documents**. We are using:

*    kill_nq from BERGEN’s built-in dataset configurations, a question answering dataset with linked documents from Wikipedia.
*   No retrievers or rerankers.